# NeighborhoodCollection population feature space

This notebook creates a `NeighborhoodCollection`, calculates a neighborhood-by-population feature space, and stores the result as a MuData modality. It downloads an example `.h5ad` from the Broad Institute Celldega supporting-data repository on Hugging Face at runtime.

In [ ]:
from pathlib import Path
from urllib.parse import quote

import requests
import scanpy as sc

import celldega as dega

In [ ]:
REPO_ID = "broadinstitute/Celldega_Supporting_Data"
REVISION = "main"
CACHE_DIR = Path("data/celldega_supporting_data")
H5AD_PATH = "Xenium_Prime_Human_Skin_FFPE_outs.h5ad"


def download_repo_file(repo_id, repo_path, cache_dir, revision="main"):
    cache_dir.mkdir(parents=True, exist_ok=True)
    local_path = cache_dir / Path(repo_path).name
    if local_path.exists():
        return local_path

    encoded_path = quote(repo_path)
    url = f"https://huggingface.co/datasets/{repo_id}/resolve/{revision}/{encoded_path}"
    with requests.get(url, stream=True, timeout=60) as response:
        response.raise_for_status()
        with local_path.open("wb") as handle:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    handle.write(chunk)
    return local_path


local_h5ad = download_repo_file(REPO_ID, H5AD_PATH, CACHE_DIR, REVISION)
adata = sc.read_h5ad(local_h5ad)
adata

In [ ]:
def ensure_spatial_coordinates(adata):
    if "spatial" in adata.obsm:
        return

    coordinate_pairs = [
        ("x_centroid", "y_centroid"),
        ("x_location", "y_location"),
        ("x", "y"),
        ("X_centroid", "Y_centroid"),
    ]
    for x_col, y_col in coordinate_pairs:
        if x_col in adata.obs and y_col in adata.obs:
            adata.obsm["spatial"] = adata.obs[[x_col, y_col]].to_numpy()
            return

    raise ValueError("The AnnData needs adata.obsm['spatial'] or x/y centroid columns.")


def pick_population_column(adata):
    candidates = [
        "cell_type",
        "celltype",
        "predicted_cell_type",
        "cell_type_major",
        "leiden",
        "cluster",
    ]
    for column in candidates:
        if column in adata.obs:
            return column
    raise ValueError(f"Could not find a population column. Available obs columns: {list(adata.obs.columns)}")


ensure_spatial_coordinates(adata)
population_col = pick_population_column(adata)
population_col

In [ ]:
gdf_hex = dega.nbhd.generate_hextile(adata, diameter=100)
gdf_hex.head()

In [ ]:
nbhd = dega.nbhd.NeighborhoodCollection(
    gdf=gdf_hex,
    nbhd_type="hextile",
    adata=adata,
)

population = nbhd.construct_population_space(
    category=population_col,
    output="percentage",
    min_cells=5,
)

population

In [ ]:
print(nbhd.collection_type)
print(list(nbhd.mod))
print(population.shape)
population.to_df().head()

The `population` modality is stored inside the same collection object. The canonical observation axis is `nbhd.obs`, while `nbhd.mod["population"].X` is the clusterable neighborhood-by-population matrix.